## Read the JSON file

In [1]:
import pandas as pd
import numpy as np

pd.options.display.float_format = "{:.2f}".format
df = pd.read_json("../data/auto.json")  

## Enrich the dataframe

In [2]:
sample = df[["CarNumber","Make","Model"]].sample(200, replace=True, random_state=21)
sample["Refund"] = df["Refund"].sample(200, replace=True, random_state=21).values
sample["Fines"]  = df["Fines"].sample(200, replace=True, random_state=21).values
concat_rows = pd.concat([df, sample], ignore_index=True)

## Enrich the concat_rows

In [3]:
np.random.seed(21)
years = pd.Series(np.random.randint(1980, 2020, size=len(concat_rows)), name="Year")
fines = pd.concat([concat_rows, years], axis=1)

## Owners + append/delete + joins

In [4]:
surn = pd.read_json("../data/surname.json")["surname"]\
        .astype(str).str.replace(r"[^A-Za-z\- ]", "", regex=True)

uniq_cars = fines["CarNumber"].dropna().unique()

owners = pd.DataFrame({
    "CarNumber": uniq_cars,
    "SURNAME": surn.sample(len(uniq_cars), replace=True, random_state=21).values
})

## Append 5 rows to fines

In [5]:
new_fines = pd.DataFrame([
    {"CarNumber":"A111AA77RUS","Make":"Ford","Model":"Focus","Refund":1.0,"Fines":1500.0,"Year":2005},
    {"CarNumber":"B222BB77RUS","Make":"Toyota","Model":"Corolla","Refund":2.0,"Fines":3200.0,"Year":2010},
    {"CarNumber":"C333CC77RUS","Make":"BMW","Model":"X5","Refund":1.0,"Fines":7000.0,"Year":2018},
    {"CarNumber":"D444DD77RUS","Make":"Audi","Model":"A4","Refund":0.0,"Fines":5000.0,"Year":2012},
    {"CarNumber":"E555EE77RUS","Make":"Kia","Model":"Rio","Refund":1.0,"Fines":800.0,"Year":1999},
])
fines2 = pd.concat([fines, new_fines], ignore_index=True)

## Update owners: drop last 20, add 3 different cars

In [6]:
owners2 = owners.iloc[:-20].copy()

new_owners = pd.DataFrame([
    {"CarNumber":"Z111ZZ99RUS","SURNAME":"Smith"},
    {"CarNumber":"Y222YY99RUS","SURNAME":"Johnson"},
    {"CarNumber":"X333XX99RUS","SURNAME":"Brown"},
])
owners2 = pd.concat([owners2, new_owners], ignore_index=True)

## Joins (4 required)

In [7]:
both = fines2.merge(owners2, on="CarNumber", how="inner")
all_cars = fines2.merge(owners2, on="CarNumber", how="outer")
only_fines = fines2.merge(owners2, on="CarNumber", how="left")
only_owners = fines2.merge(owners2, on="CarNumber", how="right")

## 5. Pivot table


In [8]:
pivot = fines2.pivot_table(
    index=["Make", "Model"],
    columns="Year",
    values="Fines",
    aggfunc="sum"
)
pivot

Year                   1980      1981      1982     1983      1984      1985  \
Make       Model                                                               
Audi                    NaN       NaN       NaN      NaN       NaN       NaN   
           A4           NaN       NaN       NaN      NaN       NaN       NaN   
BMW                     NaN       NaN       NaN      NaN       NaN       NaN   
           X5           NaN       NaN       NaN      NaN       NaN       NaN   
Ford       Focus   62394.59 395589.17 140383.76 63100.00 111294.59 189583.76   
           Mondeo       NaN       NaN       NaN      NaN       NaN       NaN   
Kia        Rio          NaN       NaN       NaN      NaN       NaN       NaN   
Skoda      Octavia 12494.59       NaN   6900.00 11594.59   1200.00  10294.59   
Toyota     Camry   18500.00   8594.59       NaN  7200.00       NaN       NaN   
           Corolla      NaN       NaN   2000.00      NaN       NaN       NaN   
Volkswagen              NaN       NaN       NaN      NaN       NaN       NaN   
           Golf    30900.00       NaN       NaN  8594.59    300.00  24000.00   
           Jetta        NaN       NaN       NaN      NaN       NaN       NaN   
           Passat       NaN   4600.00       NaN  3200.00  10000.00   5000.00   
           Touareg      NaN       NaN       NaN      NaN       NaN   5800.00   
Volvo                   NaN       NaN   6800.00      NaN       NaN       NaN   

Year                   1986      1987     1988      1989  ...      2010  \
Make       Model                                          ...             
Audi                    NaN       NaN      NaN       NaN  ...       NaN   
           A4           NaN       NaN      NaN       NaN  ...       NaN   
BMW                     NaN       NaN      NaN       NaN  ...       NaN   
           X5           NaN       NaN      NaN       NaN  ...       NaN   
Ford       Focus   88994.59 121900.00 95989.17 115500.00  ... 120183.76   
           Mondeo       NaN       NaN      NaN   8600.00  ...       NaN   
Kia        Rio          NaN       NaN      NaN       NaN  ...       NaN   
Skoda      Octavia   600.00  26700.00      NaN  91400.00  ...   3100.00   
Toyota     Camry        NaN       NaN      NaN  22400.00  ...       NaN   
           Corolla 16000.00   8000.00      NaN   4000.00  ...  27200.00   
Volkswagen          7400.00       NaN      NaN   7900.00  ...       NaN   
           Golf         NaN   9300.00      NaN   8100.00  ...       NaN   
           Jetta        NaN       NaN      NaN       NaN  ...       NaN   
           Passat  15000.00  12300.00      NaN       NaN  ...   2800.00   
           Touareg      NaN       NaN      NaN       NaN  ...   6300.00   
Volvo                   NaN       NaN      NaN       NaN  ...       NaN   

Year                   2011      2012      2013      2014      2015     2016  \
Make       Model                                                               
Audi                    NaN       NaN       NaN       NaN       NaN      NaN   
           A4           NaN   5000.00       NaN       NaN       NaN      NaN   
BMW                     NaN   3000.00       NaN       NaN   8594.59      NaN   
           X5           NaN       NaN       NaN       NaN       NaN      NaN   
Ford       Focus   86689.17 112700.00 145494.59 158894.59 203694.59 72594.59   
           Mondeo       NaN  34400.00       NaN       NaN       NaN 46200.00   
Kia        Rio          NaN       NaN       NaN       NaN       NaN      NaN   
Skoda      Octavia   500.00    500.00  13794.59   1900.00  46394.59   300.00   
Toyota     Camry        NaN  16094.59       NaN       NaN       NaN 13000.00   
           Corolla  8594.59       NaN       NaN       NaN       NaN      NaN   
Volkswagen              NaN       NaN       NaN       NaN       NaN      NaN   
           Golf      600.00       NaN   9300.00       NaN   2300.00      NaN   
           Jetta        NaN       NaN       NaN       NaN       NaN      NaN   
        

## 6. Saving fines and owners dataframes


In [9]:
fines2.to_csv("../data/fines.csv", index=False)
owners2.to_csv("../data/owners.csv", index=False)